In [0]:
%sql
CREATE OR REPLACE TEMP VIEW sellers_typed AS
SELECT
  seller_id
  , seller_zip_code_prefix
  , seller_city
  , seller_state
  , _ingested_at
FROM workspace.bronze.sellers
QUALIFY ROW_NUMBER() OVER (PARTITION BY seller_id ORDER BY _ingested_at DESC) = 1;

CREATE OR REPLACE TEMP VIEW sellers_validated AS
SELECT *,
  CASE
    WHEN seller_id IS NULL THEN 'llave invalida'
    WHEN seller_zip_code_prefix IS NULL OR seller_zip_code_prefix NOT RLIKE '^[0-9]{5}$' THEN 'codigo postal invalido'
  END AS motivo
FROM sellers_typed;

CREATE OR REPLACE TABLE workspace.silver.sellers
AS
SELECT * EXCEPT(motivo) FROM sellers_validated
WHERE motivo IS NULL;

CREATE OR REPLACE TABLE workspace.silver.sellers_quarantine
AS
SELECT * FROM sellers_validated
WHERE motivo IS NOT NULL;